In [ ]:
# Install dependencies
!pip uninstall -y -q torch torchvision torchaudio cudf easyocr numba-cuda dask-cuda pylibraft-cu12 raft-dask-cu12 pylibcudf-cu12 rmm-cu12 quantecon cuvs-cu12 ucxx-cu12 cuml-cu12 cuda-python cuml libcuml libcuvs libraft numba ydata-profiling google-colab
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch>=2.1.0 transformers>=5.15.0 datasets>=2.14.0 numpy>=1.24.0 matplotlib>=3.7.0 scipy>=1.10.0 papermill>=2.7.0 testbook>=0.4.2 diptest>=0.11.0
!pip install -q flash-linear-attention==0.4.2 --no-build-isolation
!pip install -q -U transformers


In [ ]:
import os
import sys
import time
import traceback
import torch
# Fix allocator fragmentation (exp16r OOM: reserved-but-unallocated
# grew to 1.1GB across alphas) — set BEFORE any CUDA use.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# KAGGLEHUB_CACHE: persist models across sessions
os.environ["KAGGLEHUB_CACHE"] = "/kaggle/working/.model_cache"
os.makedirs("/kaggle/working/.model_cache", exist_ok=True)
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.expanduser("~/.kaggle_test")
sys.path.insert(0, WORKING)
num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    device_L, device_S = "cuda:0", "cuda:1"
    dtype_L = dtype_S = torch.float32
elif num_gpus == 1:
    device_L = device_S = "cuda:0"
    dtype_L = torch.float16
    dtype_S = torch.float32
else:
    device_L = device_S = "cpu"
    dtype_L = dtype_S = torch.float32
if num_gpus > 0:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch {torch.__version__} | L={device_L}, S={device_S}")
RES = os.path.join(WORKING, "results")
os.makedirs(RES, exist_ok=True)
# Resolve models: kernel_sources (prev commit outputs) -> working -> HF
def resolve_model(hf_id):
    model_dir = "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b"
    # 1. Check each slug in /kaggle/input/ — listdir only (isdir lies on mounts)
    input_dir = "/kaggle/input"
    try:
        slugs = os.listdir(input_dir)
        print(f"resolve {hf_id}: slugs={slugs}")
        for slug in slugs:
            p_sub = os.path.join(input_dir, slug)
            try:
                contents = os.listdir(p_sub)
            except OSError:
                continue
            if model_dir in contents:
                p = os.path.join(p_sub, model_dir)
                print(f"Found {hf_id} at {p} (via {slug})")
                return p
    except Exception as e:
        print(f"DEBUG: /kaggle/input/ check failed: {e}")
    # 2. Check WORKING
    wf = os.path.join(WORKING, model_dir)
    if os.path.isdir(wf):
        print(f"Found {hf_id} at {wf}")
        return wf
    # 3. Return HF ID - will download + persist
    print(f"Need to download {hf_id} from HF")
    return hf_id
# Load + save for persistence (calls from_pretrained, then save_pretrained if from HF)
def load_model(cls, hf_id, **kw):
    path = resolve_model(hf_id)
    model = cls.from_pretrained(path, **kw)
    # Persist if loaded from HF
    if path == hf_id:
        dst = os.path.join(WORKING, "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b")
        if not os.path.isdir(dst):
            print(f"Saving {hf_id} to {dst}...")
            model.save_pretrained(dst)
            ensure_tokenizer(hf_id, dst)
            print(f"Saved")
    return model
# Ensure tokenizer files exist in a persisted model dir
def ensure_tokenizer(hf_id, dst):
    import transformers
    if not os.path.exists(os.path.join(dst, "tokenizer.json")):
        print(f"Saving tokenizer for {hf_id}...")
        transformers.AutoTokenizer.from_pretrained(hf_id).save_pretrained(dst)
SMALL = resolve_model("Qwen/Qwen3.5-0.8B")
LARGE = resolve_model("Qwen/Qwen3.5-4B")


In [ ]:
import contextlib
import logging
import torch
from transformers import DynamicCache
from transformers.cache_utils import (  # type: ignore[import-not-found]
    DynamicLayer,
    LinearAttentionLayer,
)
def compute_jacobian_for_tokens(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    prefix_cache=None,
) -> torch.Tensor:
    """Compute Jacobian of logits w.r.t. hidden state at layer_idx.
    Args:
        model: Qwen3_5ForCausalLM instance.
        layer_idx: Layer index (0-based) to compute Jacobian at.
        input_ids: [1, S] token IDs.
        attention_mask: [1, S] attention mask.
        token_set: list of token IDs to compute gradients for.
    Returns:
        J: [K, D] Jacobian matrix (K = len(token_set), unnormalized).
    """
    model.eval()
    S = input_ids.shape[1]
    # Constraint: single sequence, no padding. The replay phase builds causal
    # masks from the cache length with attention_mask=None, so any padding
    # token in the prefix would be silently treated as valid context.
    if attention_mask is not None and not bool(attention_mask.all()):
        raise ValueError(
            "compute_jacobian_for_tokens: only unpadded single sequences are "
            "supported (the replay phase cannot honor a padding mask)."
        )
    if prefix_cache is None:
        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
    # Match model dtype to avoid mismatch in decoder (fp16 models on GPU)
    model_dtype = next(model.parameters()).dtype
    h = h.to(model_dtype).detach().requires_grad_(True)
    # Re-enable grad: the decoder forward must build a graph over the leaf
    # h AND the row loop below must run autograd.grad with grad mode ON,
    # even when the caller is inside torch.no_grad() (e.g. eval loops in
    # exp18/19). A no_grad caller otherwise gets "element 0 ... does not
    # have a grad_fn" — the rows loop is part of the same enable_grad block.
    with torch.enable_grad():
        logits = _run_decoder_pass(model, prefix_cache, h, S, layer_idx)
        J = _compute_jacobian_rows(logits, h, token_set)
    return J
def _run_prefix_pass(
    model, input_ids: torch.Tensor, attention_mask: torch.Tensor
) -> DynamicCache:
    """Prefix pass: all tokens except last -> cache."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids[:, :-1],
            attention_mask=attention_mask[:, :-1],
            use_cache=True,
            output_hidden_states=False,
        )
    return outputs.past_key_values
def _extract_hidden_state(model, input_ids, attention_mask, layer_idx) -> torch.Tensor:
    """Full pass: extract hidden state at layer_idx for last token."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )
    # hidden_states[0] = embeddings, [l+1] = output of layer l
    h = outputs.hidden_states[layer_idx + 1][:, -1:, :]
    return h  # [1, 1, D]
def _deep_clone_cache(cache):
    """Deep clone cache, ensuring all tensors are independent copies.
    Must copy every per-layer state field, not just the obvious tensors. Both
    layer types swallow unknown constructor kwargs, so passing
    ``keys=`` / ``conv_states=`` / ``recurrent_states=`` silently discards the
    state and yields an empty cache. We therefore assign attributes directly:
    * DynamicLayer (full attention): keys / values / is_initialized / dtype /
      device. Without this, ``update()`` lazy-initialises an empty cache and
      each layer attends only to the current token.
    * LinearAttentionLayer (GDN): conv_states / recurrent_states plus the flag
      dicts (``has_previous_state`` etc.). Without these, GDN layers treat the
      new token as the first in the sequence and drop all history.
    All cloned state tensors are .detach()-ed: the prefix cache is pure data
    (built under no_grad). Detaching drops pointless CopyBackwards graph nodes
    in the enable_grad decode. The transformers 5.15 in-place cache updates
    (update_recurrent_state's copy_, causal_conv1d_update's state update —
    cudagraph static-address optimisation) are NOT fixed by the detach —
    the in-place copy_ lands on the kernel's saved input regardless. Those
    are handled by _clone_gdn_states_on_entry (clone at kernel entry so the
    read object and the in-place-written object are different objects).
    Extra unknown attributes are deep-copied as a forward-compat safety net.
    Cache OBJECT-level attributes (e.g. ``offloading`` on 5.15.1; 4.x carried
    ``_seen_tokens``) are copied from ``vars(cache)`` after the layer loop:
    a fresh ``DynamicCache()`` would reset them. On 5.15.1 ``get_seq_length()``
    is derived from ``keys.shape`` (not an object field), so the post-clone
    assert below passes on both paths — on a version that tracks sequence
    length in cache state it turns a silent Jacobian corruption into a loud
    error.
    ``copy.deepcopy(prefix_cache)`` DOES produce independent copies on 5.15.1
    (tensor values are not shared). _deep_clone_cache exists for (a) detached
    tensors (no spurious CopyBackwards in the enable_grad decode) and
    (b) robustness to per-layer attribute renames — not because deepcopy
    shares storage. Neither clone fixes the in-place write into a graph-saved
    node; that is _clone_gdn_states_on_entry's job (see its docstring).
    """
    import copy as _copy
    def _cl(t):
        return t.clone().detach() if t is not None else None
    new_cache = DynamicCache()
    for idx in range(len(cache.layers)):
        layer = cache.layers[idx]
        if isinstance(layer, DynamicLayer):
            # Deliberately NOT a vars()-walk: full-attention state fields are
            # stable across versions, and a rename would fail LOUD
            # (AttributeError), not silently corrupt the decode. The GDN
            # branch below IS generic, because GDN state fields have
            # demonstrably drifted between versions/backends.
            nl = DynamicLayer()
            nl.keys = _cl(layer.keys)
            nl.values = _cl(layer.values)
            nl.is_initialized = layer.is_initialized
            nl.dtype = getattr(layer, "dtype", None)
            nl.device = getattr(layer, "device", None)
            for k, v in vars(layer).items():
                if k not in vars(nl):
                    setattr(nl, k, _copy.deepcopy(v))
            new_cache.layers.append(nl)
        else:
            # GDN (LinearAttentionLayer). Clone every state field dynamically
            # rather than hardcoding transformers-5.15.1 attribute names
            # (number_of_states, conv_states, ...): other transformers versions
            # on other backends (e.g. the Kaggle TPU torch 2.8 image) name some
            # differently, and a hardcoded name crashes with AttributeError.
            # Tensors (or dicts of tensors) are clone+detached; scalars and
            # flag-dicts are deep-copied. This is the generic form of the old
            # field-by-field copy and behaves identically on 5.15.1.
            nl = LinearAttentionLayer()
            for k, v in vars(layer).items():
                if isinstance(v, torch.Tensor):
                    setattr(nl, k, _cl(v))
                elif isinstance(v, dict) and v and all(
                    isinstance(x, torch.Tensor) for x in v.values()
                ):
                    setattr(nl, k, {kk: _cl(xv) for kk, xv in v.items()})
                else:
                    setattr(nl, k, _copy.deepcopy(v))
            new_cache.layers.append(nl)
    # Cache object-level state (offloading flag on 5.15.1; 4.x: _seen_tokens):
    # a fresh DynamicCache() would reset it. ``layers`` is rebuilt above;
    # list-valued attrs are skipped to avoid aliasing the original.
    for k, v in vars(cache).items():
        if k == "layers" or isinstance(v, list):
            continue
        setattr(new_cache, k, _copy.deepcopy(v))
    # Guard: the clone must report the same sequence length (loud, not
    # silent, on a version that tracks it in cache state).
    assert new_cache.get_seq_length() == cache.get_seq_length()
    return new_cache
@contextlib.contextmanager
def _clone_gdn_states_on_entry():
    """Clone GDN kernel state inputs at entry: read object != written object.
    transformers 5.15 Qwen3_5GatedDeltaNet.forward resolves the kernels as
    free variables of its own module (modeling_qwen3_5), so patching the
    module attributes works — verified on 5.15.1 (calls at modeling L457/
    L506). On
    a version where GDN forward binds the kernels by a different route
    (an alias imported into another module), the patch silently no-ops and
    the in-place autograd error below resurfaces LOUD — that is the
    detection signal; there is no silent degradation.
    The recurrent loop saves the read object (version 0) for backward.
    Afterwards update_recurrent_state does a stored.copy_(new) IN PLACE on
    the very same object (5.15 cudagraph static-address optimisation) →
    version 0→1 → "modified by an inplace operation" in any backward through
    the decode (create_graph not required; the clone is also a no-op safety
    net for create_graph=True graph retention). Cloning at kernel entry makes
    the loop's saved input the fresh clone; the in-place write lands on the
    cache's original (a different object) → no in-place on a saved node.
    Re-entrancy: nesting is a NO-OP (counter below) — the outer context's
    wrappers are already active and cloning is idempotent (clone-of-clone
    changes nothing). Overwriting mq.<kernel> from OUTSIDE during the patch
    window is user error and is NOT protected (the outer exit restores its
    captured orig, discarding the overwrite).
    If the Qwen3.5 module cannot be imported, a WARNING is logged ONCE per
    interpreter (not per decode; non-Qwen3.5 builds are fine — for Qwen3.5
    with enable_grad=True the in-place autograd error is NOT protected).
    """
    global _GDN_PATCH_DEPTH
    try:
        from transformers.models.qwen3_5 import modeling_qwen3_5 as mq
    except Exception:
        if not getattr(_clone_gdn_states_on_entry, "_warned", False):
            logging.getLogger(__name__).warning(
                "GDN monkey-patch skipped (transformers.models.qwen3_5 not "
                "importable). If the model is Qwen3.5 with enable_grad=True, "
                "in-place autograd errors may occur.")
            _clone_gdn_states_on_entry._warned = True
        yield  # non-Qwen3.5 build: nothing to patch
        return
    _GDN_PATCH_DEPTH += 1
    if _GDN_PATCH_DEPTH > 1:
        # Nested: outer context already patched; wrappers chain, and the
        # clone is idempotent, so nothing to do here.
        yield
        _GDN_PATCH_DEPTH -= 1
        return
    orig_rec = mq.torch_recurrent_gated_delta_rule
    orig_conv = mq.causal_conv1d_update
    def rec(q, k, v, g, beta, *args, initial_state=None, output_final_state=True,
            use_qk_l2norm_in_kernel=False, **kw):
        # initial_state/output_final_state arrive as KEYWORDS in 5.15.1
        # (and may be positional on other versions, hence *args); clone the
        # bound state, forward everything else untouched. A kernel whose
        # signature drops these names falls through to **kw and fails loud.
        if initial_state is not None:
            initial_state = initial_state.clone()
        return orig_rec(q, k, v, g, beta, *args, initial_state=initial_state,
                        output_final_state=output_final_state,
                        use_qk_l2norm_in_kernel=use_qk_l2norm_in_kernel, **kw)
    def conv(h, cs, *args, **kw):
        # (cs, weight, [bias, activation], ...) — positional tail is
        # version-dependent; forward everything we don't own.
        if cs is not None:
            cs = cs.clone()
        return orig_conv(h, cs, *args, **kw)
    mq.torch_recurrent_gated_delta_rule = rec
    mq.causal_conv1d_update = conv
    try:
        yield
    finally:
        mq.torch_recurrent_gated_delta_rule = orig_rec
        mq.causal_conv1d_update = orig_conv
        _GDN_PATCH_DEPTH -= 1
_GDN_PATCH_DEPTH = 0
def _run_decoder_pass(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers using prefix cache."""
    # Clone cache tensors independently to avoid in-place mutation
    cache_clone = _deep_clone_cache(prefix_cache)
    # Wrapper-safe: flat CausalLM -> model.model; mm wrapper -> .language_model
    text_model = _lm_module(model)
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    layer_types = _layer_types(model)
    n_layers = _n_layers(model)
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    with _clone_gdn_states_on_entry():
        for i in range(start, n_layers):
            layer_type = layer_types[i]
            attn_mask = causal_mask_mapping[layer_type]
            x = text_model.layers[i](
                x,
                position_embeddings=position_embeddings,
                attention_mask=attn_mask,
                position_ids=text_position_ids,
                past_key_values=cache_clone,
            )
    x = text_model.norm(x)
    logits = model.lm_head(x[:, -1, :])
    return logits
def _build_causal_masks(text_model, text_pos, cache, x):
    """Build causal mask mapping for a single token with cache."""
    from transformers.masking_utils import (
        create_causal_mask,
        create_recurrent_attention_mask,
    )
    dummy_embeds = x.new_zeros(1, 1, text_model.config.hidden_size)
    mask_kwargs = {
        "config": text_model.config,
        "inputs_embeds": dummy_embeds,
        "attention_mask": None,
        "past_key_values": cache,
        "position_ids": text_pos,
    }
    return {
        "full_attention": create_causal_mask(**mask_kwargs),
        "linear_attention": create_recurrent_attention_mask(**mask_kwargs),
    }
def _compute_jacobian_rows(
    logits: torch.Tensor, h: torch.Tensor, token_set: list[int]
) -> torch.Tensor:
    """Compute Jacobian rows via ONE batched backward (is_grads_batched).
    Replaces the per-row autograd.grad loop (K backwards through the
    decoder tail) with a single vmap-style pass: grad_outputs=eye(K)
    gives all K rows at once, 3-15x faster on T4 (review point 5).
    Verified identical to the loop on synthetic linear + nonlinear graphs;
    on the real GDN decoder tail this is an EMPIRICAL property of the
    transformers 5.15 kernels (torch_recurrent_gated_delta_rule / causal
    conv1d update under vmap-style batched backward) — there is no
    per-row fallback, so a transformers upgrade that breaks this shows up
    as a runtime error, not silent degradation.
    """
    K = len(token_set)
    return torch.autograd.grad(
        outputs=logits[0, token_set],
        inputs=h,
        grad_outputs=torch.eye(K, device=h.device, dtype=h.dtype),
        is_grads_batched=True,
        retain_graph=False,
    )[0][:, 0, 0, :]
def softmax_jacobian(J: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """Gauge-invariant Jacobian: J^(p) = (diag(p) - pp^T) @ J.
    J^(p)_i = p_i * (J_i - J_bar_p),  J_bar_p = sum_k p_k * J_k
    Removes the common-mode shift (gauge) from raw Jacobian.
    """
    J_bar = (p[:, None] * J).sum(dim=0, keepdim=True)  # [1, D]
    return p[:, None] * (J - J_bar)  # [K, D]
def uniform_centered(J: torch.Tensor) -> torch.Tensor:
    """Uniform centering: J - mean(J). Removes average shift."""
    return J - J.mean(dim=0, keepdim=True)
def _last_layer_jacobian(h: torch.Tensor, norm: torch.nn.Module) -> torch.Tensor:
    """Analytical J = d h_norm / d h at the final norm (no backward passes).
    Both Qwen3_5RMSNorm and nn.LayerNorm are handled. J is evaluated at the
    norm INPUT (h), never at h_norm — evaluating the formula at the norm
    output gives a matrix with ~0.87 relative error (no longer LayerNorm-like
    statistics). Qwen3.5's final norm is Qwen3_5RMSNorm:
      h_norm = (h * rsqrt(mean(h^2) + eps) * (1 + w))
      J = r · diag(1+w) · (I - xhat xhat^T / D),  r = rsqrt(mean(h^2)+eps), xhat = r*h
    LayerNorm: J = (I - 1/D - xhat xhat^T / D) / std, rows scaled by w.
    """
    # h may arrive as [1, S, D] (full-sequence hidden_states) or [1, D]/[D].
    # The last-layer READOUT Jacobian is the FINAL-position map (it feeds the
    # final-token logits), so collapse a [1, S, D] to its last position first.
    # BUG this fixes: [1, S, D].reshape(1,-1) -> [1, S*D] -> [S*D, S*D] identity
    # (e.g. [65536, 65536] = 16 GB) -> OOM on GPU / XLA compile-abort on TPU
    # (previously misattributed to an "XLA phantom dot" shape bug).
    if h.dim() == 3:
        h = h[:, -1, :]
    x = h.reshape(1, -1).float()  # [1, D], the norm INPUT (row vector)
    D = x.shape[-1]
    eps = float(getattr(norm, "eps", None) or 1e-5)
    w = getattr(norm, "weight", None)
    dev = x.device
    if getattr(norm, "bias", None) is not None:  # LayerNorm family
        mu = x.mean(dim=-1, keepdim=True)
        std = (x.var(dim=-1, unbiased=False, keepdim=True) + eps).sqrt()
        xhat = (x - mu) / std  # [1, D]
        J = (
            torch.eye(D, device=dev)
            - torch.ones(D, D, device=dev) / D
            - xhat.T @ xhat / D
        ) / std.squeeze(0)
        if w is not None:
            J = J * w.float().squeeze(0).unsqueeze(1)  # J = diag(w) @ J
        return J
    # RMSNorm family (Qwen3_5RMSNorm, Llama-style).
    r = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    xhat = x * r  # [D]
    # y_j = (1+w_j) r x_j; J = r * diag(1+w) @ (I - xhat xhat^T / D)
    # (derived: J_ij = g_j (r d_ij - r^3 x_i x_j / D) = g_j r (d_ij - xhat_i xhat_j / D))
    g = 1.0 + w.float().squeeze(0) if w is not None else None
    M = torch.eye(D, device=dev) - xhat.T @ xhat / D
    if g is None:
        return r.squeeze(0) * M
    return r.squeeze(0) * torch.diag(g) @ M
def compute_hidden_jacobian(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
    h: torch.Tensor | None = None,
) -> torch.Tensor:
    """Compute Jacobian of normalized last hidden state w.r.t. hidden state at layer_idx.
    J_norm = d(h_norm) / d(h_{layer_idx}), shape [D, D].
    Independent of lm_head — computed once, reused across W updates.
    Then J_S = lm_head.weight[token_set] @ J_norm is differentiable through W.
    For last_layer: J_norm is LayerNorm Jacobian (analytical, no backward passes).
    Args:
        create_graph: if True, J_norm has grad_fn.
                      h is kept connected to model params (no detach). Full forward pass.
                      Costs ~D backward passes per sample.
        verbose: print progress during J_norm computation.
    """
    model.eval()
    n_layers = _n_layers(model)
    if create_graph:
        # Keep gradient graph alive: h is connected to model params (+ any hooks)
        S = input_ids.shape[1]
        # Build prefix cache FIRST — before any forward involving hooks.
        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
        # Full forward to get h at layer_idx — connected to all model params + hooks
        with torch.enable_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )
        assert outputs.hidden_states is not None  # requested
        h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D], connected!
        model_dtype = next(model.parameters()).dtype
        assert h is not None  # narrowed: param was Tensor|None
        h = h.to(model_dtype)  # preserve grad graph
        if h.grad_fn is None:
            # Fully frozen graph: no trainable path reaches this layer
            # (e.g. linear_attention pair with no LoRA patch, or a
            # no_grad caller). Make h a leaf so the decoder pass below
            # still yields a differentiable h_norm; J_norm values are
            # identical to the connected case.
            h = h.detach().requires_grad_(True)
        if layer_idx == n_layers - 1:
            with torch.enable_grad():
                h_norm = _lm_module(model).norm(h[:, -1, :])  # [1, D]
        else:
            with torch.enable_grad():
                h_norm = _run_decoder_to_norm_compat(
                    model, prefix_cache, h, S, layer_idx
                )
        # [1, 1, D] — _compute_jacobian_rows_full indexes grad[0, 0, :].
        # enable_grad: autograd.grad needs grad mode ON even when the
        # caller is inside torch.no_grad().
        with torch.enable_grad():
            J_norm = _compute_jacobian_rows_full(
                h_norm, h, create_graph=True, verbose=verbose
            )
        return J_norm
    # Last layer: analytical norm Jacobian, no backward passes.
    if layer_idx == n_layers - 1:
        with torch.no_grad():
            if h is None:
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    use_cache=False,
                )
                assert outputs.hidden_states is not None  # requested
                h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]
            assert h is not None
            h = h.to(next(model.parameters()).dtype)
        return _last_layer_jacobian(h, _lm_module(model).norm)
    # General case: decode through remaining layers + norm
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    if h is None:
        h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
    model_dtype = next(model.parameters()).dtype
    h = h.to(model_dtype).detach().requires_grad_(True)
    # Re-enable grad: h_norm must carry a grad_fn over h even if the caller
    # is inside torch.no_grad() (e.g. eval loops in exp16r/exp17). The row
    # loop below (autograd.grad) ALSO needs grad mode ON — a no_grad caller
    # otherwise gets "element 0 ... does not have a grad_fn".
    with torch.enable_grad():
        h_norm = _run_decoder_to_norm_compat(model, prefix_cache, h, S, layer_idx)
        J_norm = _compute_jacobian_rows_full(
            h_norm, h, create_graph=create_graph, verbose=verbose
        )
    return J_norm
def _run_decoder_to_norm(
    text_model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers, return norm(x) without lm_head."""
    cache_clone = _deep_clone_cache(prefix_cache)
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    n_layers = text_model.config.num_hidden_layers
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    with _clone_gdn_states_on_entry():
        for i in range(start, n_layers):
            layer_type = text_model.config.layer_types[i]
            attn_mask = causal_mask_mapping[layer_type]
            x = text_model.layers[i](
                x,
                position_embeddings=position_embeddings,
                attention_mask=attn_mask,
                position_ids=text_position_ids,
                past_key_values=cache_clone,
            )
    return text_model.norm(x[:, -1, :])
def _compute_jacobian_rows_full(
    output: torch.Tensor,
    h: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
) -> torch.Tensor:
    """Full [D, D] Jacobian via autograd.grad."""
    import time as _time
    D = h.shape[-1]
    out_flat = output.flatten()
    grads = []
    # create_graph=True: every row must retain the graph — the caller
    # backpropagates a final loss through J_norm afterwards (exp17 bridge
    # loss -> o_proj HVP). retain=(i<D-1) would free the (second-order)
    # graph after the last row, so the caller's first backward would fail
    # with "backward through the graph a second time". Memory is bounded
    # by the refresh schedule (one graph per si, freed at the next refresh
    # / pair end).
    start = _time.time()
    for i in range(D):
        grad = torch.autograd.grad(
            outputs=out_flat[i],
            inputs=h,
            retain_graph=(True if create_graph else i < D - 1),
            create_graph=create_graph,
        )[0]
        grads.append(grad[0, 0, :])
        if verbose and i % 128 == 0:
            elapsed = _time.time() - start
            rate = i / elapsed if elapsed > 0 else 999
            eta = elapsed * (D / i) - elapsed if i > 0 else 0.0
            print(f"        J_norm: {i}/{D} ({rate:.0f}/s, eta={eta:.0f}s)")
    J = torch.stack(grads)
    elapsed_total = _time.time() - start
    if verbose:
        print(f"        J_norm done: {D}/{D} in {elapsed_total:.1f}s")
    return J
def _lm_module(model):
    """The text model with .layers/.norm/.rotary_emb.
    Flat CausalLM: model.model IS the text model. Multimodal wrapper
    (Qwen3_5ForConditionalGeneration): model.model.language_model.
    """
    mm = model.model
    if hasattr(mm, "layers"):
        return mm
    return mm.language_model
def _n_layers(model) -> int:
    """num_hidden_layers for flat or wrapper configs."""
    cfg = model.config
    return int(
        cfg.num_hidden_layers
        if hasattr(cfg, "num_hidden_layers")
        else cfg.text_config.num_hidden_layers
    )
def _layer_types(model) -> list:
    """layer_types for flat or wrapper configs."""
    cfg = model.config
    return list(
        cfg.layer_types if hasattr(cfg, "layer_types") else cfg.text_config.layer_types
    )
def _run_decoder_to_norm_compat(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """_run_decoder_to_norm over _lm_module (flat or nested model)."""
    return _run_decoder_to_norm(_lm_module(model), prefix_cache, h, S, layer_idx)
def h_norm_at_perturbed_h(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    h: torch.Tensor,
) -> torch.Tensor:
    """h_norm after the prefix, with the l_S hidden state pinned to `h`.
    Finite-diff support (B2 spot check): call twice, h and h + eps*e_j,
    and difference the two h_norms. h shape [1, 1, D] (any device/dtype);
    the pinning is done on a copy — the input is not mutated.
    """
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    with torch.no_grad():
        return _run_decoder_to_norm_compat(model, prefix_cache, h, S, layer_idx)
def _forward_h_norm(
    model,
    layer_idx,
    input_ids,
    attention_mask,
    h: torch.Tensor | None = None,
):
    """Forward once -> (h, h_norm), h graph-connected at layer l_S.
    Shared by compute_hidden_jacobian_rows and bridge_rows_backward (A4):
    the prefix pass runs before the model forward (hook-safe), h is the
    l_S hidden state (kept in graph so a caller that patched o_proj / lm_head
    gets HVP/bridge grads from the downstream backward), h_norm = RMSNorm(h)
    (last layer) or decoder-to-norm (otherwise).
    h (optional): reuse a pre-computed l_S hidden state instead of doing a
    fresh full forward (v23.1: exp17's KL forward already produced it, so
    the refresh reuses it and only runs the cheap decoder-tail-to-norm).
    """
    model.eval()
    n_layers = _n_layers(model)
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    with torch.enable_grad():
        if h is None:
            # standalone: full forward to obtain h (original A4 behavior)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )
            assert outputs.hidden_states is not None  # requested
            h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]
        assert h is not None
        h = h.to(next(model.parameters()).dtype)
        if h.grad_fn is None:
            h = h.detach().requires_grad_(True)
        if layer_idx == n_layers - 1:
            h_norm = _lm_module(model).norm(h[:, -1, :])  # [1, D]
        else:
            h_norm = _run_decoder_to_norm_compat(
                model, prefix_cache, h, S, layer_idx
            )
    return h, h_norm
def bridge_rows_backward(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    p: torch.Tensor,
    W_rows: torch.Tensor,
    J_tgt_p: torch.Tensor,
    extra_loss: torch.Tensor | None = None,
    chunk_size: int = 8,
    verbose: bool = False,
    h: torch.Tensor | None = None,
) -> tuple[torch.Tensor, float]:
    """v23 A4: chunked bridge backward (the v22 OOM fix).
    Forward once + J_bar once, then build each chunk of K rows and backprop
    its (1 - cos) bridge loss immediately (extra_loss — the KL — is
    backproped with the first chunk). Backproping per chunk frees that
    chunk's 2nd-order row-graphs at once, so peak = chunk_size + 1 (the
    shared J_bar) second-order graphs, not K. Per-chunk gradients are
    identical to a single backward on the total (1 - cos).mean() loss.
    Gradients reach W_rows (live lm_head + B_lh @ A_lh) and o_proj (HVP
    through h). Returns (J_S_p [K, D] detached, bridge_loss scalar).
    h (optional): reuse a pre-computed l_S hidden state (v23.1) so this
    does NOT redo the full forward — only the cheap decoder-tail-to-norm
    runs. exp17's KL forward already produced the same h.
    """
    import time as _time
    K = len(token_set)
    h, h_norm = _forward_h_norm(
        model, layer_idx, input_ids, attention_mask, h=h
    )
    def _row_grad(w: torch.Tensor) -> torch.Tensor:
        return torch.autograd.grad(
            outputs=h_norm @ w,
            inputs=h,
            retain_graph=True,
            create_graph=True,
        )[0][0, 0, :]
    start = _time.time()
    J_bar = _row_grad(p @ W_rows)  # shared; freed by the last chunk's bwd
    # extra_loss (the KL) gets a SEPARATE backward before the chunk loop.
    # retain_graph=True keeps the shared h-graph alive for the chunks while
    # the KL's OWN forward graph (a separate forward in exp17) frees here.
    # We deliberately do NOT add it to chunk 0: in a shared-forward test/
    # wiring that would free the h-graph the chunks still need.
    if extra_loss is not None:
        extra_loss.backward(retain_graph=True)
    chunks: list[torch.Tensor] = []
    total_bridge = 0.0
    for c in range(0, K, chunk_size):
        hi = min(c + chunk_size, K)
        is_last = (c + chunk_size) >= K
        rows_c = torch.stack(
            [p[i] * (_row_grad(W_rows[i]) - J_bar) for i in range(c, hi)]
        )
        tgt_c = J_tgt_p[c:hi].float()
        cos_c = torch.nn.functional.cosine_similarity(rows_c.float(), tgt_c, dim=1)
        chunk_loss = (1.0 - cos_c).sum() / K
        # retain_graph on all-but-last keeps the ONE shared h forward graph
        # alive so the next chunk can still build its rows; the chunk's
        # 2nd-order row-graph is released by refcounting once we del it
        # (forward nodes stay, referenced by h / h_norm / J_bar).
        chunk_loss.backward(retain_graph=not is_last)
        total_bridge += chunk_loss.item()
        chunks.append(rows_c.detach())
        del rows_c, cos_c, chunk_loss, tgt_c
        if verbose:
            print(
                f"        bridge_rows: {hi}/{K} "
                f"({hi / max(_time.time() - start, 1e-6):.1f} chunks/s)"
            )
    return torch.cat(chunks), total_bridge
def compute_hidden_jacobian_rows(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    p: torch.Tensor,
    W_rows: torch.Tensor,
    create_graph: bool = False,
    chunk_size: int = 8,
    verbose: bool = False,
) -> torch.Tensor:
    """v23 A1: per-row J_S_p = p_i*(J_i - J_bar), no dense [D,D] J_norm.
    K-subset softmax gauge — the SAME gauge as the exp16 target (p_s =
    softmax(full_logits[token_set])) and the exp17 dense bridge
    (softmax_jacobian(W[token_set]@J_norm, p_L)):
      p = re-normalized dist over the K-subset [K], W_rows = W[token_set]
      [K, D]; J_bar = d(h_norm @ (p @ W_rows))/dh; row_i =
      p[i] * (d(h_norm @ W_rows[i])/dh - J_bar).
    h stays graph-connected (via _run_decoder_to_norm_compat, which
    routes through _lm_module so both flat CausalLM and the multimodal
    wrapper work), so a caller that patched the target layer's o_proj
    and/or lm_head gets its HVP/bridge gradients from the downstream
    loss backward. chunk_size (A4) bounds the 2nd-order graphs alive
    at once; create_graph=True retains them for a later loss backward,
    create_graph=False frees each chunk after the gradient (for cheap
    no-grad refreshes / evals).
    """
    model.eval()
    n_layers = _n_layers(model)
    S = input_ids.shape[1]
    h, h_norm = _forward_h_norm(model, layer_idx, input_ids, attention_mask)
    return _compute_jacobian_rows_matrixfree(
        h_norm,
        h,
        W_rows,
        p,
        token_set,
        create_graph=create_graph,
        chunk_size=chunk_size,
        verbose=verbose,
    )
def _compute_jacobian_rows_matrixfree(
    h_norm: torch.Tensor,
    h: torch.Tensor,
    W_rows: torch.Tensor,
    p: torch.Tensor,
    token_set: list[int],
    create_graph: bool = True,
    chunk_size: int = 8,
    verbose: bool = False,
) -> torch.Tensor:
    """J_S_p [K, D] rows, K-subset softmax gauge, no dense [D,D] J_norm.
    p [K] = re-normalized softmax over the K-subset (exp16/exp17 gauge),
    W_rows [K, D] = the selected rows. J_bar = d(h_norm @ (p @ W_rows))/dh;
    row_i = p[i]*(d(h_norm @ W_rows[i])/dh - J_bar). Identical to
    softmax_jacobian(W_rows @ J_norm, p) but with K backward passes
    instead of a retained [D,D] 2nd-order matrix (v22 OOM).
    Chunked (A4): only chunk_size row-graphs alive at once.
    """
    import time as _time
    K = len(token_set)
    def _row_grad(w: torch.Tensor) -> torch.Tensor:
        return torch.autograd.grad(
            outputs=h_norm @ w,
            inputs=h,
            retain_graph=True,
            create_graph=create_graph,
        )[0][0, 0, :]
    start = _time.time()
    J_bar = _row_grad(p @ W_rows)  # [D]; p@W_rows = effective selected row
    rows = []
    for c in range(0, K, chunk_size):
        for local, i in enumerate(range(c, min(c + chunk_size, K))):
            row = p[i] * (_row_grad(W_rows[i]) - J_bar)
            rows.append(row)
        if verbose:
            done = min(c + chunk_size, K)
            el = _time.time() - start
            print(f"        J_S_p: {done}/{K} ({done / max(el, 1e-6):.1f}/s)")
    out = torch.stack(rows)
    elapsed = _time.time() - start
    if verbose:
        print(f"        J_S_p done: {K}/{K} in {elapsed:.1f}s")
    return out


In [ ]:
import hashlib
import os
import torch
_gramian_cache: dict[str, torch.Tensor] = {}
_GRAMIAN_DIR = os.path.join("results", ".gramians")
def jacobian_gap(
    J_L: torch.Tensor,
    J_S: torch.Tensor,
    E_L: torch.Tensor,
    E_S: torch.Tensor,
    probs_L: torch.Tensor,
    threshold: float = 0.0,
) -> float:
    """Compute the Jacobian gap (alignment loss, LOWER = BETTER) between large and small models.
    Terminology (v4.0): gap = weighted (1 - cos), lower=better; bridge = weighted
    cos, higher=better; gap = 1 - bridge at equal weights. Callers reporting
    "bridge" (E-16a/b) invert: bridge = 1 - jacobian_gap(...). The legacy
    `jacobian_bridge` alias (which silently returned the GAP direction despite
    its name) was removed after being identified as an inversion trap — the
    inversion lives in the experiment kernels (kaggle_kernel/kernel-exp1*/...py),
    not here.
    The bridge chain: J_L @ E_L^T @ E_S @ (E_S^T @ E_S)^{-1}.
    The inverse Gramian (E_S^T @ E_S)^{-1} (disk-cached, see _inverse_gramian)
    corrects for non-orthogonal embeddings, so identity (J_L == J_S,
    E_L == E_S) yields gap ~0.
    threshold: row mask on BOTH norms, (n1 > threshold) & (n2 > threshold).
    Default 0.0 = sentinel-only mode: only exactly-zero rows are masked
    (no decay filtering). Historical default was 1e-6; the paper's
    relative quantile tau_l = 0.01 * median_i ||J_S[i]|| (E-6 calibration,
    ~1e-4 at L18) can be passed explicitly to enable decay masking.
    """
    G_inv = _inverse_gramian(E_S)
    J_L_proj = _project_to_vocab(J_L, E_L)
    J_S_target = _project_to_small(J_L_proj, E_S, G_inv)
    mask = _valid_mask(J_S, J_S_target, threshold)
    if mask.sum() == 0:
        return 1.0
    return _weighted_cosine_loss(J_S, J_S_target, probs_L, mask)
def _project_to_vocab(J_L: torch.Tensor, E_L: torch.Tensor) -> torch.Tensor:
    """Project J_L [K, D_L] through E_L^T -> [K, V]."""
    return J_L.float() @ E_L.float().T
def _project_to_small(
    J_L_proj: torch.Tensor, E_S: torch.Tensor, G_inv: torch.Tensor
) -> torch.Tensor:
    """Project [K, V] through E_S @ G_inv -> [K, D_S].
    G_inv = (E_S^T @ E_S)^{-1} corrects non-orthogonal embeddings.
    """
    return J_L_proj @ E_S.float() @ G_inv
def _inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1}, cached on disk per embedding hash."""
    key = _gramian_key(E)
    if key not in _gramian_cache:
        path = os.path.join(_GRAMIAN_DIR, f"{key}.pt")
        if os.path.exists(path):
            _gramian_cache[key] = torch.load(path, weights_only=True)
        else:
            _gramian_cache[key] = _compute_inverse_gramian(E)
            os.makedirs(_GRAMIAN_DIR, exist_ok=True)
            torch.save(_gramian_cache[key], path)
    return _gramian_cache[key].to(E.device)
def _gramian_key(E: torch.Tensor) -> str:
    """Stable hash of an embedding matrix for disk caching.
    Hashes shape/device header + first, last, and 10 deterministic mid-rows
    (uniformly spaced, seeded from row count so the sample is stable across
    runs but covers the interior). First/last rows alone would collide for
    matrices that share boundary rows (e.g. PAD/UNK) but differ internally.
    Hashing the full 248k×1024 matrix would be prohibitively expensive.
    """
    V = E.shape[0]
    idx = [0, V - 1]
    if V > 4:
        step = (V - 1) // 10
        idx += sorted({(i * step) % (V - 1) + 1 for i in range(1, 10)})
    header = f"{tuple(E.shape)}|{str(E.device)}".encode()
    data = header + E[0].cpu().numpy().tobytes() + E[-1].cpu().numpy().tobytes()
    for i in sorted(set(idx)):
        data += E[i].cpu().numpy().tobytes()
    h = hashlib.sha256(data).hexdigest()[:16]
    return f"g_{h}"
def _compute_inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1} via pseudoinverse for numerical stability."""
    G = E.float().T @ E.float()
    return torch.linalg.pinv(G)
def _valid_mask(
    J_S: torch.Tensor, J_target: torch.Tensor, threshold: float
) -> torch.Tensor:
    """Mask rows where either norm <= threshold (sentinel rows).
    threshold=0.0 (default): only exactly-zero rows are masked.
    threshold>0: decay masking (absolute threshold; the paper's relative
    tau_l = 0.01*median is applied by the caller, E-6 calibration).
    """
    n1 = J_S.float().norm(dim=1)
    n2 = J_target.float().norm(dim=1)
    return (n1 > threshold) & (n2 > threshold)
def _weighted_cosine_loss(
    J_S: torch.Tensor,
    J_target: torch.Tensor,
    probs: torch.Tensor,
    mask: torch.Tensor,
) -> float:
    """Weighted cosine dissimilarity loss."""
    cos_sim = torch.nn.functional.cosine_similarity(
        J_S.float()[mask], J_target[mask], dim=1
    )
    w = probs[mask]
    w = w / (w.sum() + 1e-8)
    return (w * (1.0 - cos_sim)).sum().item()


In [ ]:
import torch
def _validate(jacobians: list[torch.Tensor]) -> None:
    if not jacobians:
        raise ValueError("jacobians: empty list")
    ref = jacobians[0]
    for J in jacobians[1:]:
        if J.shape != ref.shape:
            raise ValueError(f"jacobians: shape mismatch {tuple(J.shape)} vs {tuple(ref.shape)}")
        if J.device != ref.device:
            raise ValueError(f"jacobians: device mismatch {J.device} vs {ref.device}")
def compute_esg(
    jacobians: list[torch.Tensor],
    n_components: int = 32,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute ESG from a list of Jacobian matrices.
    Pooling is INTRA-TRAJECTORY: all J_i must be positions of ONE sample
    (document). Cross-sample pooling is invalid — see module docstring /
    paper §15 (E-0.1); aggregate across samples at the scalar-metric level.
    Args:
        jacobians: list of N Jacobian matrices (one trajectory), each [K, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: [min(n_components, K)] sorted descending (K < n_components
        possible in degenerate cases; in practice K = |𝒮| ≥ 32).
        eigenvectors: [min(n_components, K), K] corresponding eigenvectors.
    """
    _validate(jacobians)
    G = _build_gramian(jacobians)
    eigenvalues, eigenvectors = _spectral_decomposition(G, n_components)
    return eigenvalues, eigenvectors
def compute_esg_per_sample(
    sample_jacobians: list[list[torch.Tensor]],
    n_components: int = 32,
) -> tuple[list[torch.Tensor], list[torch.Tensor]]:
    """Sample-local ESG: one Gramian per sample, NO cross-sample pooling.
    Args:
        sample_jacobians: list over samples; each entry is a list of the
            sample's per-position Jacobians [K, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: list [n_samples] of [n_components].
        eigenvectors: list [n_samples] of [n_components, K].
    Cross-sample aggregation happens on scalars (lambda, PR, SNR, r_eff),
    never on the matrices (E-0.1: cross-sample J's are near-orthogonal).
    """
    if not sample_jacobians:
        raise ValueError("sample_jacobians: empty list")
    eigenvalues = []
    eigenvectors = []
    for Js in sample_jacobians:
        ev, evec = compute_esg(Js, n_components=n_components)
        eigenvalues.append(ev)
        eigenvectors.append(evec)
    return eigenvalues, eigenvectors
def _build_gramian(jacobians: list[torch.Tensor]) -> torch.Tensor:
    """G = (1/N) * sum_i (J_i @ J_i^T)  (intra-trajectory pool)."""
    U = jacobians[0].shape[0]
    device = jacobians[0].device
    G = torch.zeros(U, U, device=device, dtype=torch.float)
    for J in jacobians:
        G += J.float() @ J.float().T
    G /= len(jacobians)
    return G
def _spectral_decomposition(
    G: torch.Tensor, n_components: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Eigen decomposition, sorted descending."""
    eigenvalues, eigenvectors = torch.linalg.eigh(G)
    # Sort descending
    idx = eigenvalues.argsort(descending=True)
    eigenvalues = eigenvalues[idx][:n_components]
    eigenvectors = eigenvectors[:, idx][:, :n_components].T
    return eigenvalues, eigenvectors


In [ ]:
import warnings
from collections import Counter
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
def uniform_layer_mapping(n_layers_large: int, n_layers_small: int) -> dict[int, int]:
    """Uniform layer mapping from large to small model layers."""
    mapping = {}
    for i in range(n_layers_large):
        j = round(i * (n_layers_small - 1) / (n_layers_large - 1))
        mapping[i] = j
    return mapping
def get_layer_types(model: AutoModelForCausalLM) -> list[str]:
    """Get layer types from model config (full_attention / linear_attention)."""
    cfg = model.config
    return list(
        cfg.layer_types if hasattr(cfg, "layer_types") else cfg.text_config.layer_types
    )
_DOMAIN_DATASETS = {
    "wiki": ("allenai/c4", "en", "train", "text"),
    "math": ("nvidia/OpenMathInstruct-2", None, "train", "problem"),
    "code": ("neyralabs/apex-agents", None, "train", "prompt"),
    "chat": ("HuggingFaceH4/ultrachat_200k", None, "train_sft", "messages"),
}
def _load_dataset_for_domain(
    domain: str,
) -> tuple[str, str | None, str, str]:
    """Return (dataset_id, config, split, text_field) for a domain.
    Falls back to wiki (C4) if the target dataset is unavailable, with a
    RuntimeWarning: a fallback run silently poisons cross-domain comparisons
    (E-13/E-27 T_domain), so the substitution must be visible in the log.
    """
    ds_id, cfg, split, field = _DOMAIN_DATASETS[domain]
    try:
        kw: dict[str, object] = {"split": split, "streaming": True}
        if cfg is not None:
            kw["name"] = cfg
        ds = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
        next(iter(ds))  # force the first fetch: streaming is lazy, a broken/
        # gated dataset would otherwise pass the check and fail inside
        # load_calibration_data (loud crash instead of the intended fallback).
        return ds_id, cfg, split, field
    except Exception:
        warnings.warn(
            f"domain {domain} ({ds_id}) unavailable — falling back to wiki/C4. "
            f"Domain labels in this run are INVALID for cross-domain "
            f"comparisons (E-13/E-27).",
            RuntimeWarning,
            stacklevel=2,
        )
        return _DOMAIN_DATASETS["wiki"]
def _extract_text_from_sample(
    sample: dict,
    field: str,
) -> str:
    """Extract text from a dataset sample, handling different formats."""
    if field == "messages":
        # Chat format: list of {role, content}
        texts = [m.get("content", "") for m in sample.get("messages", [])]
        return " ".join(texts)
    val = sample.get(field, "")
    if isinstance(val, list):
        # e.g. code_contests solutions: list of strings
        return "\n".join(str(v) for v in val)
    if isinstance(val, dict):
        return str(val)
    return val
def load_calibration_data(
    model_name: str,
    max_length: int = 64,
    min_length: int = 64,
    n_samples: int = 4,
    domain: str = "wiki",
    save_offset_mapping: bool = False,
) -> list[dict[str, torch.Tensor]]:
    """Load and tokenize calibration data.
    Defaults match the tested protocol (paper §12/C.6: S=64, filter >= 64,
    N=4); experiments pass explicit lengths. domain: 'wiki' (C4), 'math'
    (OpenMathInstruct-2), 'code' (apex-agents), 'chat' (ultrachat). Falls back
    to wiki if the target dataset is unavailable. save_offset_mapping: adds
    `offset_mapping` to each sample (E-31 sentence boundaries).
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    ds_id, cfg, split, field = _load_dataset_for_domain(domain)
    kw: dict[str, object] = {"split": split, "streaming": True}
    if cfg is not None:
        kw["name"] = cfg
    dataset = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
    samples = []
    for raw in dataset:
        text = _extract_text_from_sample(raw, field)
        if not text:
            continue
        if save_offset_mapping:
            toks = tokenizer(
                text, max_length=max_length, truncation=True,
                add_special_tokens=False, return_offsets_mapping=True,
            )
            if len(toks["input_ids"][0]) >= min_length:
                samples.append({
                    "input_ids": toks["input_ids"][0],
                    "offset_mapping": torch.tensor(
                        toks["offset_mapping"][0], dtype=torch.int32
                    ),
                })
        else:
            tokens = tokenizer(
                text, max_length=max_length, truncation=True,
                return_tensors="pt",
            )
            if tokens["input_ids"].shape[1] >= min_length:
                samples.append(tokens)
        if len(samples) >= n_samples:
            break
    return samples
def build_global_token_set(
    model: AutoModelForCausalLM,
    calibration_samples: list[dict[str, torch.Tensor]],
    K: int = 64,
) -> list[int]:
    """Build global token set from top-K logits across all samples."""
    token_counter: Counter[int] = Counter()
    model.eval()  # type: ignore[attr-defined]
    device = next(model.parameters()).device  # type: ignore[attr-defined]
    with torch.no_grad():
        for sample in calibration_samples:
            inputs = {
                "input_ids": sample["input_ids"].to(device),
                "attention_mask": sample["attention_mask"].to(device),
            }
            outputs = model(**inputs)  # type: ignore[operator]
            logits = outputs.logits[:, -1, :]
            top_idx = logits.topk(K, dim=-1).indices[0]
            token_counter.update(top_idx.tolist())
    U = [tid for tid, _ in token_counter.most_common(K)]
    return sorted(U)
def load_models(
    large_name: str, small_name: str, dtype: torch.dtype = torch.float32
) -> tuple[torch.nn.Module, torch.nn.Module]:
    """Load large and small models with eager attention.
    dtype defaults to float32: the tested protocols run on CPU, where fp16
    is ~10x slower (paper §12/C.9). Pass torch.float16 explicitly for GPU.
    Thread count (torch.set_num_threads(4) in §12/C.9) is set by the caller's
    setup, not here.
    """
    kwargs = {
        "torch_dtype": dtype,
        "attn_implementation": "eager",
    }
    large = AutoModelForCausalLM.from_pretrained(large_name, **kwargs)
    large.eval()  # type: ignore[attr-defined]
    small = AutoModelForCausalLM.from_pretrained(small_name, **kwargs)
    small.eval()  # type: ignore[attr-defined]
    return large, small
def extract_embeddings(
    model: AutoModelForCausalLM,
) -> torch.Tensor:
    """Extract embedding matrix from model.
    The bridge projects through the AN-embedding matrix W (§3/§5). For the
    tied-weight checkpoints in scope, W = E (embed_tokens). If a checkpoint
    is loaded WITHOUT tied weights, W = lm_head.weight ≠ E and the projection
    would silently use the wrong matrix. We therefore warn ONCE per model if
    lm_head.weight is not the same storage as embed_tokens.weight — a loud,
    non-fatal signal that the W = E assumption does not hold for this model.
    """
    emb = model.model.embed_tokens.weight.detach()  # type: ignore[attr-defined]
    _warn_if_not_tied(model, emb)
    return emb
def _warn_if_not_tied(model: AutoModelForCausalLM, emb: torch.Tensor) -> None:
    lm_head = getattr(getattr(model, "lm_head", None), "weight", None)
    if lm_head is None:
        return
    if not lm_head.data_ptr() == emb.data_ptr():
        # data_ptr identity covers tied weights; fall back to value equality
        # for the (rare) case of a tied-but-copied view.
        if not torch.equal(lm_head.detach(), emb):
            warnings.warn(
                f"extract_embeddings: lm_head.weight is NOT tied to "
                f"embed_tokens.weight (model={type(model).__name__}). The "
                f"bridge projects through E assuming W = E (§3); with untied "
                f"weights the an-embedding matrix is lm_head.weight and the "
                f"gap/bridge numbers are silently wrong. Extract lm_head "
                f"explicitly if this is intended.",
                UserWarning,
                stacklevel=2,
            )


In [ ]:
import os
# Max depth under input_root to search (producer puts cache at depth 2:
# results/targets; 4 covers any reasonable layout).
_MAX_DEPTH = 4
def _has_pt(d: str) -> bool:
    try:
        return any(f.endswith(".pt") for f in os.listdir(d))
    except OSError:
        return False
def _walk(root: str, depth: int):
    """Yield dirs under root (listdir-based), depth-limited."""
    try:
        entries = sorted(os.listdir(root))
    except OSError:
        return
    for e in entries:
        p = os.path.join(root, e)
        if _has_pt(p):
            yield p
        if depth > 1:
            yield from _walk(p, depth - 1)
def find_targets_dir(preferred: str, input_root: str = "/kaggle/input") -> str:
    """Return the dir holding the target-cache *.pt files.
    Order: `preferred` (local path / runner-injected), then a listdir walk
    of every mount under `input_root`. Raises FileNotFoundError with the
    mount inventory when nothing is found (no silent fallback, no listdir crash).
    """
    if _has_pt(preferred):
        return preferred
    for hit in _walk(input_root, _MAX_DEPTH):
        return hit
    mounts = []
    try:
        for d in sorted(os.listdir(input_root)):
            try:
                mounts.append(f"{d}/:{os.listdir(os.path.join(input_root, d))[:4]}")
            except OSError:
                mounts.append(f"{d}/:<unreadable>")
    except OSError:
        mounts.append("<no input mount>")
    raise FileNotFoundError(
        f"No target cache .pt in {preferred} nor under {input_root}/ "
        f"(mounts: {mounts}). Run Stage 0 first — it must publish results/targets/*.pt."
    )


In [ ]:
from __future__ import annotations
def _uniform_layer_mapping(n_L: int, n_S: int) -> dict[int, int]:
    """Uniform large->small layer mapping: l_L -> round(l_L*(n_S-1)/(n_L-1)).
    Inlined copy of prism.calibration.uniform_layer_mapping (identical formula)
    so this module stays standalone for kernel inlining.
    """
    return {i: round(i * (n_S - 1) / (n_L - 1)) for i in range(n_L)}
def get_layer_types(model) -> list[str]:
    """Per-layer attention type from model.config.layer_types.
    Qwen3.5 style hybrid: entries 'full_attention' or 'linear_attention'.
    `model.config` (post-AutoModel load) exposes `layer_types` at top level
    (Qwen3_5TextConfig). Returns lowercase list.
    """
    types = getattr(model.config, "layer_types", None)
    if types is None:
        # Fallback: nested text_config (defensive; Qwen3_5 exposes top-level).
        tc = getattr(model.config, "text_config", None)
        types = getattr(tc, "layer_types", None)
    if types is None:
        raise RuntimeError(
            "model.config has no layer_types (and no text_config.layer_types)"
        )
    return [str(t) for t in types]
def select_pairs(n_L: int, n_S: int, layer_types: list[str]) -> list[tuple[int, int]]:
    """Return (l_L, l_S) pairs: full-attn l_S + 0_0 + last, sorted by l_S."""
    if n_L < 1 or n_S < 1 or len(layer_types) != n_S:
        raise ValueError(
            f"invalid: n_L={n_L}, n_S={n_S}, layer_types={len(layer_types)}"
        )
    mapping = _uniform_layer_mapping(n_L, n_S)
    full_S = {i for i, t in enumerate(layer_types) if t == "full_attention"}
    selected = {0, n_S - 1} | full_S
    pairs = [_nearest_preimage(mapping, j) for j in sorted(selected)]
    if (n_L - 1, n_S - 1) not in pairs:
        pairs.append((n_L - 1, n_S - 1))
    return pairs
def _nearest_preimage(mapping: dict[int, int], j: int) -> tuple[int, int]:
    """l_L = argmin |mapping[l] - j|, tie -> smaller l (the selected l_S is j)."""
    best_l, best_d = 0, float("inf")
    for li in range(len(mapping)):
        d = abs(mapping[li] - j)
        if d < best_d:
            best_l, best_d = li, d
    return (best_l, j)
def classify_pair(l_S: int, layer_types: list[str]) -> str:
    """'bridge' if small-model layer l_S has o_proj (full_attention), else 'kl_only'."""
    return (
        "bridge"
        if l_S < len(layer_types) and layer_types[l_S] == "full_attention"
        else "kl_only"
    )


In [ ]:
import math
import json
import os
import sys
import time
import traceback
import numpy as np
import torch
from scipy.stats import t as tdist
S = 128
D = 1024
LAYERS = (6, 12, 18, 19, 22, 23)  # sorted: monotone depth profile
T_FULL = 128
EPS_REL = np.round(np.logspace(np.log10(0.05), np.log10(0.70), 16), 6)
NEPS = len(EPS_REL)
EPS_CANON = 0.15
R_SWEEP = (1, 2, 4, 8, 16)
BOOT_N = 64
PERM_N = 1000
MODEL = 'Qwen/Qwen3.5-0.8B'
OUT = 'results/exp32r3_layer_profile.json'  # separate file: E-32r.2 kept
CHUNK = 4096  # review point 2/6: T4-safe (16 MB activations, 2 MB output)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# --------------------------------------------------------------------------
# math helpers (verbatim from E-32r.2)
# --------------------------------------------------------------------------
def offdiag_norm(M):
    return float(math.sqrt(max((M * M).sum() - (np.diag(M) ** 2).sum(), 0.0)))
def gauge_free_rows(J, p):
    """P(p) = diag(p) - pp^T in token space: J [K,D] rows=tokens."""
    return J * p[:, None] - (p @ J)[None, :] * p[:, None]
def top_r_right(J, r):
    """Top-r right singular vectors [r, D] (rows = directions in D-space)."""
    U, _, Vt = np.linalg.svd(J, full_matrices=False)
    return Vt[:r, :]
def _right_angle(V1, V2):
    """Largest principal angle (rad): max(acos(svdvals(V1 @ V2.T))) — same
    estimator as exp33r delta_svd."""
    sv = np.linalg.svdvals(V1 @ V2.T)
    return float(math.acos(float(np.clip(sv.max(), 0.0, 0.999999999999))))
def d3_top_r(Jseq, Pseq, r, mode, p0=None):
    """d3 = per-pair top-r largest principal angle (rad), mode raw/gf/fixP
    (p0 — canonical, passed EXPLICITLY). Returns [T-1]."""
    out = []
    for t in range(len(Jseq) - 1):
        A = Jseq[t]
        B = Jseq[t + 1]
        if mode == 'gf':
            A = gauge_free_rows(A, Pseq[t])
            B = gauge_free_rows(B, Pseq[t + 1])
        elif mode == 'fixP':
            A = gauge_free_rows(A, p0)
            B = gauge_free_rows(B, p0)
        V1 = top_r_right(A, r)
        V2 = top_r_right(B, r)
        out.append(_right_angle(V1, V2))  # single acos
    return np.array(out)
def ols_hc3(x, y, rng=None, n_perm=PERM_N):
    """OLS y ~ alpha*x + beta, HC3 standard errors; permutation p for alpha."""
    n = len(x)
    if n < 5:
        return {'alpha': None, 'beta': None, 'alpha_se': None,
                'alpha_p': None, 'perm_p': None}
    X = np.stack([np.ones(n), x], axis=1)
    XtX_inv = np.linalg.inv(X.T @ X)
    beta_hat = XtX_inv @ X.T @ y
    resid = y - X @ beta_hat
    H = X @ XtX_inv @ X.T
    lev = np.diag(H)
    hc3 = np.sqrt(np.maximum(np.diag(XtX_inv) * float(
        (resid ** 2 / np.maximum(1 - lev, 1e-12) ** 2).sum()), 0.0))
    tstat = beta_hat[1] / hc3[1] if hc3[1] > 0 else 0.0
    alpha_p = float(2 * tdist.sf(abs(tstat), n - 2))
    if rng is None:
        perm_p = None
    else:
        obs = abs(tstat)
        cnt = 0
        for _ in range(n_perm):
            yp = y[rng.permutation(n)]
            bp = np.linalg.solve(X.T @ X, X.T @ yp)
            rp = yp - X @ bp
            h3 = np.sqrt(np.maximum(np.diag(XtX_inv) * float(
                (rp ** 2 / np.maximum(1 - lev, 1e-12) ** 2).sum()), 0.0))
            if h3[1] > 0 and abs(bp[1] / h3[1]) >= obs:
                cnt += 1
        perm_p = (cnt + 1) / (n_perm + 1)
    return {'alpha': float(beta_hat[1]), 'beta': float(beta_hat[0]),
            'alpha_se': float(hc3[1]), 'alpha_p': alpha_p, 'perm_p': perm_p}
def fit_cubic(eps, y):
    """y = a*e + b*e^2 + c*e^3 (no const); returns a1,a2,a3,pC,resid."""
    e = np.asarray(eps, dtype=np.float64)
    Z = np.asarray(y, dtype=np.float64)
    A = np.stack([e, e ** 2, e ** 3], axis=1)
    coef, *_ = np.linalg.lstsq(A, Z, rcond=None)
    res = Z - A @ coef
    dof = Z.size - A.shape[1]
    s2 = float(res @ res) / dof
    se = math.sqrt(max(s2 * np.linalg.inv(A.T @ A)[2, 2], 0.0))
    tstat = coef[2] / se if se > 0 else 0.0
    return float(coef[0]), float(coef[1]), float(coef[2]), \
        float(2 * tdist.sf(abs(tstat), dof)), res
def fit_all(eps_abs, dz):
    """Vectorized cubic fit. eps_abs [T,NEPS], dz [T,K,K,NEPS] (float32 ok).
    Fit in float64 (upcast of the float32 residuals) — review point 4.
    Returns a1,a2,a3,pC [T,K,K] (float32), resid [T,K,K,NEPS] (float64).
    """
    e = eps_abs.astype(np.float64)
    Z = dz.astype(np.float64) if dz.dtype != np.float64 else dz  # float32->64
    nrm = e[:, 0] / EPS_REL[0]                                   # [T]
    er = np.asarray(EPS_REL, dtype=np.float64)
    A0 = np.stack([er, er ** 2, er ** 3], -1)                    # [NEPS,3]
    inv0 = np.linalg.inv(A0.T @ A0)
    Atb = np.einsum('eu,tije->tuij', A0, Z)                      # [T,3,K,K]
    coef_rel = np.einsum('pq,tqij->tpij', inv0, Atb)
    n1 = 1.0 / nrm
    a1 = (coef_rel[:, 0] * n1[:, None, None]).astype(np.float32)
    a2 = (coef_rel[:, 1] * (n1[:, None, None] ** 2)).astype(np.float32)
    a3 = (coef_rel[:, 2] * (n1[:, None, None] ** 3)).astype(np.float32)
    resid = Z - (a1[:, :, :, None] * e[:, None, None, :]
                 + a2[:, :, :, None] * e[:, None, None, :] ** 2
                 + a3[:, :, :, None] * e[:, None, None, :] ** 3)
    s2v = (resid ** 2).sum(-1) / max(NEPS - 3, 1)
    se_rel = np.sqrt(np.maximum(s2v * inv0[2, 2], 0.0))
    tstat = np.zeros_like(coef_rel[:, 2])
    np.divide(coef_rel[:, 2], se_rel, out=tstat, where=se_rel > 0)
    pC = (2 * tdist.sf(np.abs(tstat), NEPS - 3)).astype(np.float32)
    return a1, a2, a3, pC, resid
def pick_t0(P128, sent):
    """Canonical p0 position: non-sentinel position with p closest to median."""
    ok = ~sent
    if not ok.any():
        ok = np.ones_like(sent)
    med = np.median(P128[ok], 0)
    sel = int(np.argmin(np.linalg.norm(P128[ok] - med[None, :], axis=1)))
    return int(np.where(ok)[0][sel])
def _clean(o):
    if isinstance(o, dict):
        return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.floating, float)):
        f = float(o)
        return None if not math.isfinite(f) else f
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o
# --------------------------------------------------------------------------
# model (decoder tail on device — review point 2: .to(dev)/.cpu())
# --------------------------------------------------------------------------
def load_decoder(U128, model_dir):
    """RMSNorm + g + lm_head[U128] as float32 torch tensors on DEVICE.
    Returns (forward, W_U, g, eps_norm); forward: [n, D] f32 -> [n, K] f32.
    """
    from transformers import AutoModelForCausalLM
    torch.set_num_threads(1)  # CPU fallback only; GPU path ignores this
    model = AutoModelForCausalLM.from_pretrained(
        model_dir, dtype=torch.float32, attn_implementation='eager').eval()
    W = model.lm_head.weight.detach().float().numpy()
    g = (1 + model.model.norm.weight.detach().float()).numpy()
    eps_norm = float(model.model.norm.eps)
    W_U = W[U128]
    del model
    g_t = torch.from_numpy(g.astype(np.float32)).to(DEVICE)
    W_U_t = torch.from_numpy(W_U.astype(np.float32).T).to(DEVICE)
    @torch.no_grad()
    def forward(hs):  # hs [n, D] f32 numpy -> logits [n, K] f32 numpy
        x = torch.from_numpy(hs.astype(np.float32)).to(DEVICE)
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps_norm)
        x = x * g_t
        lg = x @ W_U_t
        return lg.cpu().numpy()
    return forward, W_U, g, eps_norm
def analytic_J(W_U, g, h):
    r = np.linalg.norm(h)
    u = h / r
    M = W_U * g[None, :]
    return math.sqrt(D) / r * (M - (M @ u)[:, None] * u[None, :])
# --------------------------------------------------------------------------
def doc_layer_metrics(forward, ck, dom_si, T):
    """One doc x one layer: all injections, fit, all TЗ metrics. T=128.
    float32 storage for dz32 (review point 4: NO float64 copy of dz32);
    fit_all upcasts to float64 internally. CHUNK=4096 GPU forward.
    """
    (dom, si) = dom_si
    Js, Ps, Raws, Hs = [], [], [], []
    for t in range(T):
        k = (dom, si, t)
        Js.append(ck['JR'][k].numpy())
        Ps.append(ck['P128'][k].numpy())
        Raws.append(ck['P128_RAW'][k].numpy())
        Hs.append(ck['H'][k].numpy())
    JR = np.stack(Js).astype(np.float64)
    P128 = np.stack(Ps).astype(np.float64)
    RAW = np.stack(Raws).astype(np.float64)
    H = np.stack(Hs).astype(np.float64)
    sent = RAW.sum(-1) < 0.5
    vp = ~sent[:-1] & ~sent[1:]
    norms = np.linalg.norm(H, axis=1)
    eps_abs = np.array(EPS_REL)[:, None] * norms[None, :]  # [NEPS, T]
    # --- injections: h'[t,i,e] = H[t] + eps_abs[e,t] * Jn[t,i] ---
    Jn = JR / np.maximum(np.linalg.norm(JR, axis=-1)[..., None], 1e-12)
    H32 = H.astype(np.float32)
    Jn32 = Jn.astype(np.float32)
    scaled = eps_abs.T.astype(np.float32)[:, None, :, None] * Jn32[:, :, None, :]
    h_inj = (H32[:, None, None, :] + scaled).reshape(-1, D)
    del scaled, Jn32, Jn
    # base_all: one forward over T positions
    outs = []
    for c0 in range(0, T, CHUNK):
        outs.append(forward(H32[c0:c0 + CHUNK]))
    base_all = np.concatenate(outs)
    del H32
    K = S  # vocab slice (logits); T — positions (Fix v2: stride/reshape by K)
    dz32 = np.empty((T, K, K, NEPS), dtype=np.float32)
    for t_ in range(T):
        hc = h_inj[t_ * K * NEPS:(t_ + 1) * K * NEPS]
        out = []
        for c0 in range(0, hc.shape[0], CHUNK):
            out.append(forward(hc[c0:c0 + CHUNK]))
        L = np.concatenate(out).reshape(K, NEPS, S).transpose(0, 2, 1)
        dz32[t_] = L - base_all[t_][None, :, None]
    del h_inj, base_all
    a1, a2, a3, pC, resid = fit_all(eps_abs.T, dz32)  # float64 fit inside
    A1 = a1 + 2.0 * a2 * (EPS_CANON * norms)[:, None, None]
    # --- s_t / slope / residual (TЗ «Математика», raw normalization) ---
    ESG_all = np.einsum('std,sjd->stj', JR, JR)
    ESGd = np.diagonal(ESG_all, axis1=1, axis2=2)
    off_esg = np.sqrt(np.maximum((ESG_all ** 2).sum((-1, -2)) - (ESGd ** 2).sum(-1), 0.0))
    nb = np.array([offdiag_norm(a2[t]) for t in range(T)])
    s_t = EPS_CANON * norms * nb / np.maximum(off_esg, 1e-12)
    # slope_ratio: E-32r.2 formula UNCHANGED (comparability): ratio of
    # |dz/eps| at the two smallest eps; 1.0 = linear regime.
    e0v = EPS_REL[0] * norms
    e1v = EPS_REL[1] * norms
    slope0 = np.abs(dz32[..., 0]) / np.maximum(e0v[:, None, None], 1e-12)
    slope1 = np.abs(dz32[..., 1]) / np.maximum(e1v[:, None, None], 1e-12)
    slope = slope0 / np.maximum(slope1, 1e-12)
    slope_ratio_med = float(np.median(slope))
    # quad_frac: NEW metric (no E-32r.2 counterpart) — relative quadratic
    # contribution at eps0. Absolute-eps reconstruction:
    # eps_abs = EPS_REL[0] * ||h_t|| (norms).
    c1 = a1 + 2.0 * a2 * (EPS_REL[0] * norms)[:, None, None]
    s1 = np.linalg.norm(a1, axis=-1)
    s2 = np.linalg.norm(c1, axis=-1)
    quad = (s1 - s2) / np.maximum(np.abs(s2), 1e-12)
    quad_frac_med = float(np.median(quad))
    c_sig = float((pC[~sent] < 0.05).mean())
    # Fix (d): a1 vs ESG (Frobenius cosine) — fit sanity check, ~1 expected
    amed = np.median(a1, 0)
    ESG_med = np.median(ESG_all, 0)
    ESGn = ESG_med / np.maximum(
        np.sqrt(np.maximum(np.diag(ESG_med), 0.0))[None, :], 1e-12)
    a_vs_esg = float((amed * ESGn).sum() / max(
        np.linalg.norm(amed) * np.linalg.norm(ESGn), 1e-12))
    resid_frac_t = np.zeros((T, NEPS))  # (T, NEPS), Fix v3 — E-32r.2 semantics:
    # ||resid_f64|| / ||dz_measured||; float32 norm is a denominator, no
    # catastrophic cancellation, so float32 storage loses nothing here.
    for e in range(NEPS):
        num = np.linalg.norm(resid[:, :, :, e], axis=(1, 2))
        den = np.linalg.norm(dz32[:, :, :, e], axis=(1, 2))
        resid_frac_t[:, e] = num / np.maximum(den, 1e-12)
    rfe_all = resid_frac_t
    del dz32, resid
    # --- p0 canonical + d3 (top-r, sweep); p0 NORMALIZED (review point 6) ---
    t0 = pick_t0(P128, sent)
    p0 = P128[t0] / np.maximum(P128[t0].sum(), 1e-30)
    d3 = {m: {r: d3_top_r(JR, P128, r, m, p0) for r in R_SWEEP}
          for m in ('raw', 'gf', 'fixP')}
    # --- Q1/Q4 by |dp| over valid pairs (review #2: n>=16/group) ---
    dp = np.linalg.norm(np.diff(P128, axis=0), axis=1)
    dpv = dp[vp]
    q1 = q4 = None
    mQ1 = mQ4 = None
    bQ1, bQ4 = None, None
    db_rel = None
    if len(dpv) >= 32:
        q1, q4 = np.percentile(dpv, [25, 75])
        Q1 = np.where((dp <= q1) & vp)[0]
        Q4 = np.where((dp >= q4) & vp)[0]
        if len(Q1) >= 16 and len(Q4) >= 16:
            mQ1 = np.median(a2[Q1], 0)
            mQ4 = np.median(a2[Q4], 0)
            bQ1, bQ4 = float(np.linalg.norm(mQ1)), float(np.linalg.norm(mQ4))
            b_mean = float(np.linalg.norm(np.median(a2[:-1][vp], 0)))
            db_rel = float(np.linalg.norm(mQ4 - mQ1) / max(b_mean, 1e-12))
    # --- OLS+HC3: nb(t) ~ alpha*|dp(t)| + beta, valid pairs ---
    tidx = np.where(vp)[0]
    ols = ols_hc3(dp[tidx], nb[tidx]) if len(tidx) >= 5 else None
    # --- 4 cross-checks (TЗ): rho(nb, dSVD_raw/fixP/gf/r=1, |dp|) ---
    def spearman(x, y):
        n = len(x)
        if n < 3:
            return float('nan')
        rx = np.argsort(np.argsort(x)) - (np.arange(n) + 1) / (n + 1)
        ry = np.argsort(np.argsort(y)) - (np.arange(n) + 1) / (n + 1)
        return float(np.corrcoef(rx, ry)[0, 1])
    rho = {
        'rho_nb_dSVD_raw1': spearman(nb[tidx], d3['raw'][1][tidx]),
        'rho_nb_dSVD_fixP1': spearman(nb[tidx], d3['fixP'][1][tidx]),
        'rho_nb_dSVD_gf1': spearman(nb[tidx], d3['gf'][1][tidx]),
        'rho_nb_dp': spearman(nb[tidx], dp[tidx]),
    }
    out = {
        't0': int(t0),
        's_l_median': float(np.median(s_t)),
        's_l_iqr': [float(np.percentile(s_t, 25)), float(np.percentile(s_t, 75))],
        'slope_ratio_median': slope_ratio_med,
        'slope_ratio_iqr': [float(np.percentile(slope, 25)),
                            float(np.percentile(slope, 75))],
        'quad_frac_median': quad_frac_med,
        'quad_frac_iqr': [float(np.percentile(quad, 25)),
                          float(np.percentile(quad, 75))],
        'A1_median': float(np.median(A1)),
        'nb_median': float(np.median(nb[~sent])),
        'frac_c_sig': c_sig,
        'resid_frac_by_eps_med': [float(np.median(resid_frac_t[:, e]))
                                  for e in range(NEPS)],
        'resid_frac_by_eps_max': [float(np.max(resid_frac_t[:, e]))
                                  for e in range(NEPS)],
        'sentinel_fraction': float(sent.mean()),
        'n_valid_pairs': int(vp.sum()),
        'a_vs_ESG_cos': a_vs_esg,
        'd3_by_r': {m: {str(r): [float(np.median(d3[m][r][vp])),
                                [float(np.percentile(d3[m][r][vp], 25)),
                                 float(np.percentile(d3[m][r][vp], 75))]]
                        for r in R_SWEEP} for m in ('raw', 'gf', 'fixP')},
        'Q1_n': int(len(np.where((dp <= (q1 if q1 is not None else 0)) & vp)[0]))
                if q1 is not None else 0,
        'Q4_n': int(len(np.where((dp >= (q4 if q4 is not None else 0)) & vp)[0]))
                if q4 is not None else 0,
        'bQ1_fro': bQ1,
        'bQ4_fro': bQ4,
        'delta_b_rel': db_rel,
        'ols': ols,
        'cross_checks': rho,
    }
    return _clean(out), {'dp_v': dp[vp], 'nb_v': nb[:-1][vp]}
def _pooled_ols(cache, keys):
    """Pooled OLS+HC3: all valid pairs of all docs of one layer."""
    x_all = [cache[k]['dp_v'] for k in keys]
    y_all = [cache[k]['nb_v'] for k in keys]
    if sum(len(a) for a in x_all) < 5:
        return None
    x = np.concatenate(x_all)
    y = np.concatenate(y_all)
    return ols_hc3(x, y, rng=np.random.default_rng(99), n_perm=PERM_N)
def main_32r3layerprofile(results_dir=None):
    global S, D, NEPS, EPS_REL  # selfcheck temporarily changes globals
    import argparse
    results_dir = results_dir or os.path.join(os.path.dirname(os.getcwd()),
                                              'results')
    if not os.path.isdir(results_dir):
        results_dir = 'results'
    ap = argparse.ArgumentParser()
    ap.add_argument('--out', default=os.path.join(results_dir,
                                                  'exp32r3_layer_profile.json'))
    ap.add_argument('--layers', type=int, nargs='+', default=list(LAYERS))
    ap.add_argument('--cache', default=None, help='E-27b dense cache dir')
    ap.add_argument('--model', default=None, help='decoder-tail model dir')
    ap.add_argument('--doc-cap', type=int, default=None)
    ap.add_argument('--selfcheck', action='store_true')
    a = ap.parse_args()
    # cache/model resolution: args -> /kaggle/input (kernel sources) ->
    # /kaggle/working -> local paths
    cache_dir = a.cache
    if cache_dir is None and os.path.isdir('/kaggle/input'):
        for slug in sorted(os.listdir('/kaggle/input')):
            p = os.path.join('/kaggle/input', slug, 'results')
            if os.path.isfile(os.path.join(p, f'exp27b_jcache_L{LAYERS[0]}.pt')):
                cache_dir = p
                print(f'cache from kernel source: {p}', flush=True)
                break
    if cache_dir is None:
        for cand in ('/kaggle/working/results',
                     os.path.expanduser('~/work/prism/results'),
                     '/data/prism_kernels/exp27b/results'):
            if os.path.isfile(os.path.join(cand, f'exp27b_jcache_L{LAYERS[0]}.pt')):
                cache_dir = cand
                break
    model_dir = a.model
    if model_dir is None and os.path.isdir('/kaggle/input'):
        for slug in sorted(os.listdir('/kaggle/input')):
            p = os.path.join('/kaggle/input', slug, 'qwen35-08b')
            if os.path.isdir(p):
                model_dir = p
                print(f'model from kernel source: {p}', flush=True)
                break
    if model_dir is None:
        for cand in (os.path.join(
            '/kaggle/working', 'qwen35-08b'),
            os.path.expanduser('~/.cache/hub/qwen35-08b')):
            if os.path.isdir(cand):
                model_dir = cand
                break
        if model_dir is None:
            model_dir = MODEL  # HF download
    print(f'cache: {cache_dir}', flush=True)
    print(f'model: {model_dir}', flush=True)
    if a.selfcheck:
        rng = np.random.default_rng(0)
        # (1) fit_cubic 16-point decode
        y = 1.0 * EPS_REL + 2.0 * EPS_REL ** 2 + 0.5 * EPS_REL ** 3 + \
            1e-6 * rng.standard_normal(NEPS)
        a1f, a2f, a3f, pc, _ = fit_cubic(EPS_REL, y)
        assert abs(a1f - 1) < 0.05 and abs(a2f - 2) < 0.05 and abs(a3f - 0.5) < 0.05, \
            f'fit16 fail {a1f} {a2f} {a3f}'
        # (2) fit_all == pairwise, incl. float32 input (review point 4)
        rng3 = np.random.default_rng(2)
        Sa, Ta = 3, 4
        eabs = rng3.uniform(10, 30, (Sa, 1)) * np.asarray(EPS_REL)
        dz = rng3.standard_normal((Sa, Ta, Ta, NEPS)).astype(np.float32)
        a1v, a2v, a3v, pCv, _ = fit_all(eabs, dz)
        for s_ in range(Sa):
            for i_ in range(Ta):
                for j_ in range(Ta):
                    aa, ab, ac, pcc, _ = fit_cubic(eabs[s_], dz[s_, i_, j_])
                    assert abs(a1v[s_, i_, j_] - aa) < 1e-3 and abs(a2v[s_, i_, j_] - ab) < 1e-3
        # (3) OLS+HC3: signal alpha + p-value + permutation
        rngx = np.random.default_rng(4)
        x = rngx.uniform(0, 1, 40)
        yx = 3.0 * x + 1.0 + 0.05 * rngx.standard_normal(40)
        o = ols_hc3(x, yx, rng=rngx, n_perm=200)
        assert abs(o['alpha'] - 3.0) < 0.3 and o['alpha_p'] < 0.001, f'ols fail {o}'
        assert o['perm_p'] < 0.05
        # (4) d3_top_r NUMERIC (Fix e): identical -> 0, orthogonal -> pi/2
        J1 = np.zeros((8, 6)); J1[0, 0] = 1.0
        J2 = np.zeros((8, 6)); J2[0, 1] = 2.0
        Jseq = np.stack([J1, J1, J2])
        Pc = np.full((3, 8), 1.0 / 8.0)
        d = d3_top_r(Jseq, Pc, 1, 'raw')
        assert d.shape == (2,)
        assert abs(d[0]) < 1e-4, f'id angle {d[0]}'
        assert abs(d[1] - math.pi / 2) < 1e-6, f'orth angle {d[1]}'
        dfix = d3_top_r(Jseq, Pc, 1, 'fixP', np.full(8, 1.0 / 8.0))
        assert np.all(np.isfinite(dfix)) and abs(dfix[0]) < 1e-4, f'fixP id {dfix}'
        rngt = np.random.default_rng(11)
        Jfr = np.stack([rngt.standard_normal((8, 6)), rngt.standard_normal((8, 6))])
        Pseq1h = np.zeros((2, 8)); Pseq1h[:, 0] = 1.0
        d_implicit = d3_top_r(Jfr, Pseq1h, 1, 'fixP', Pseq1h[0])
        d_explicit = d3_top_r(Jfr, Pseq1h, 1, 'fixP', np.full(8, 1.0 / 8.0))
        assert abs(d_implicit[0] - d_explicit[0]) > 1e-6, \
            f'p0 not respected {d_implicit} vs {d_explicit}'
        # (5) quad_frac formula, [T,K,K] shapes: quadratic-free -> quad == 0
        # exactly; linear-free (a1=0) -> quad == -1 (s1=0, s2>0)
        Tz = np.array([10.0, 20.0, 30.0])
        a1z = np.zeros((3, 4, 4)); a2z = np.zeros((3, 4, 4))
        c1z = a1z + 2.0 * a2z * (EPS_REL[0] * Tz)[:, None, None]
        assert c1z.shape == (3, 4, 4), c1z.shape
        assert np.allclose(
            (np.linalg.norm(a1z, axis=-1) - np.linalg.norm(c1z, axis=-1))
            / np.maximum(np.abs(np.linalg.norm(c1z, axis=-1)), 1e-12), 0.0)
        a1q = np.zeros((3, 4, 4)); a2q = np.ones((3, 4, 4))
        c1q = a1q + 2.0 * a2q * (EPS_REL[0] * Tz)[:, None, None]
        assert np.allclose(
            (np.linalg.norm(a1q, axis=-1) - np.linalg.norm(c1q, axis=-1))
            / np.maximum(np.abs(np.linalg.norm(c1q, axis=-1)), 1e-12), -1.0)
        # (6) pick_t0 canonical determinism
        rng2 = np.random.default_rng(1)
        Pc2 = rng2.random((20, 128))
        Pc2 = Pc2 / Pc2.sum(-1, keepdims=True)
        sentc = np.zeros(20, bool)
        sentc[3] = sentc[7] = True
        assert pick_t0(Pc2, sentc) == pick_t0(Pc2.copy(), sentc.copy())
        assert pick_t0(Pc2, sentc) not in (3, 7)
        # (7) doc_layer_metrics T=K fake-forward (linear -> slope 0, resid 0)
        S0, D0, NEPS0, EPS0 = S, D, NEPS, EPS_REL
        try:
            S, D, NEPS = 4, 16, 5
            EPS_REL = np.array([0.05, 0.1, 0.2, 0.4, 0.7])
            rng7 = np.random.default_rng(77)
            Wf = rng7.standard_normal((16, S))
            class _Arr:
                def __init__(self, aa):
                    self.a = aa
                def numpy(self):
                    return self.a
            keys7 = [(0, 0, t) for t in range(2)]
            ck7 = {
                'JR': {k: _Arr(rng7.standard_normal((S, 16))) for k in keys7},
                'P128': {k: _Arr(rng7.random(S)) for k in keys7},
                'P128_RAW': {k: _Arr(rng7.random(S) * 10) for k in keys7},
                'H': {k: _Arr(rng7.standard_normal(16)) for k in keys7},
            }
            res7, _ = doc_layer_metrics(lambda hs: hs @ Wf, ck7, (0, 0), 2)
            assert len(res7['resid_frac_by_eps_med']) == NEPS
            assert max(res7['resid_frac_by_eps_med']) < 1e-3, \
                f'T=K resid {res7["resid_frac_by_eps_med"]}'
            assert res7['a_vs_ESG_cos'] is not None
            assert 'd3_by_r' in res7 and math.isfinite(res7['s_l_median'])
        finally:
            S, D, NEPS, EPS_REL = S0, D0, NEPS0, EPS0
        print('selfcheck OK', flush=True)
        return
    if cache_dir is None:
        print('[FAIL] no E-27b dense cache found; pass --cache')
        return
    t0 = time.time()
    # --- cache load + L19 sanity assert (review point 1) ---
    ck = {}
    for l in a.layers:
        p = os.path.join(cache_dir, f'exp27b_jcache_L{l}.pt')
        ck[l] = torch.load(p, map_location='cpu', weights_only=False)
        docs_l = {(k[0], k[1]) for k in ck[l]['JR'].keys()}
        ts_l = {k[2] for k in ck[l]['JR'].keys()}
        assert len(docs_l) == 32, f'L{l}: expected 32 docs, got {len(docs_l)}'
        assert ts_l == set(range(T_FULL)), \
            f'L{l}: expected t=0..{T_FULL - 1}, got {len(ts_l)} distinct'
    U128 = np.array(ck[a.layers[0]]['U128'], dtype=np.int64)
    for l in a.layers[1:]:
        assert np.array_equal(np.array(ck[l]['U128'], dtype=np.int64), U128), \
            f'L{l}: U128 mismatch'
    # P128_RAW: identical across layers (decoder-tail p128, layer-independent)
    # — the L19 file predates the RAW addition; fill from the first layer
    # that has it.
    raw_src = None
    for l in a.layers:
        if ck[l].get('P128_RAW'):
            raw_src = l
            break
    if raw_src is None:
        print('[FAIL] P128_RAW missing in every layer cache')
        return
    for l in a.layers:
        if not ck[l].get('P128_RAW'):
            ck[l]['P128_RAW'] = ck[raw_src]['P128_RAW']
            print(f'L{l}: P128_RAW filled from L{raw_src} ' \
                  f'(identical across layers)', flush=True)
        assert set(ck[l]['P128_RAW'].keys()) == set(ck[l]['JR'].keys()), \
            f'L{l}: P128_RAW keys mismatch'
    docs = sorted({(k[0], k[1]) for k in ck[a.layers[0]]['JR'].keys()})
    if a.doc_cap:
        docs = docs[:a.doc_cap]
    print(f'docs: {len(docs)}, layers: {a.layers}, eps: {NEPS} pts, '
          f'T={T_FULL} all layers, device={DEVICE}', flush=True)
    forward, W_U, g, eps_norm = load_decoder(U128, model_dir)
    # E-35 identity (1 position, 1 doc, L23 — desync guard); skipped when
    # L23 not in --layers (it validates L23 analytics, not the pipeline)
    if 23 in ck:
        d23 = docs[0]
        h0 = np.stack([ck[23]['H'][(d23[0], d23[1], 0)].numpy()]).reshape(D)
        J0 = ck[23]['JR'][(d23[0], d23[1], 0)].numpy()
        J23a = analytic_J(W_U, g, h0)
        cos = float((J23a * J0).sum() / (np.linalg.norm(J23a) * np.linalg.norm(J0)))
        print(f'E-35 identity cos = {cos:.6f}', flush=True)
        assert abs(abs(cos) - 1) < 1e-4, 'E-35 identity fail'
    else:
        print('E-35 identity skipped: L23 not in --layers', flush=True)
    per_doc_layer = {}
    cache = {}
    # --- checkpoint (self-sourced kernel: prev /kaggle/working is mounted
    # read-only at /kaggle/input/<slug>/ — resume from there if present) ---
    ckpt_path = os.path.join(results_dir, 'exp32r3_checkpoint.pt')
    for cand in ([ckpt_path] + [
            os.path.join('/kaggle/input', s, 'results', 'exp32r3_checkpoint.pt')
            for s in sorted(os.listdir('/kaggle/input'))] if os.path.isdir('/kaggle/input') else []):
        if os.path.isfile(cand):
            c = torch.load(cand, map_location='cpu', weights_only=False)
            per_doc_layer = c.get('per_doc_layer', {})
            cache = c.get('cache', {})
            print(f'checkpoint {cand}: {len(per_doc_layer)} (doc,layer) pairs resumed',
                  flush=True)
            break
    def save_ckpt():
        torch.save({'per_doc_layer': per_doc_layer, 'cache': cache}, ckpt_path)
    first = True
    for li, l in enumerate(a.layers):
        for d in docs:
            key = f'{l}|{d[0]}/{d[1]}'
            if key in per_doc_layer:
                continue
            t1 = time.time()
            per_doc_layer[key], cache[key] = \
                doc_layer_metrics(forward, ck[l], d, T_FULL)
            save_ckpt()  # per-pair checkpoint: T4 wipe at any point is safe
            if first:
                try:
                    import psutil
                    rss = psutil.Process().memory_info().rss / 2 ** 30
                    print(f'  first (doc, layer) {time.time() - t1:.1f}s '
                          f'RSS={rss:.2f} GB', flush=True)
                except Exception:
                    print(f'  first (doc, layer) {time.time() - t1:.1f}s',
                          flush=True)
                first = False
            print(f'  L{l} T={T_FULL} {d}: {time.time() - t1:.1f}s', flush=True)
    del ck
    # --- aggregation E-34: per-doc median -> median/IQR over docs ---
    def _agg(vals):
        vals = np.array([v for v in vals if v is not None])
        if len(vals) == 0:
            return None
        return float(np.median(vals)), [float(np.percentile(vals, 25)),
                                        float(np.percentile(vals, 75))]
    layers_out = {}
    boot_rng = np.random.default_rng(12345)
    for l in a.layers:
        keys = [f'{l}|{d[0]}/{d[1]}' for d in docs]
        missing = [k for k in keys if k not in per_doc_layer]
        if missing:
            sys.exit(f'L{l}: {len(missing)} (doc,layer) pairs missing from '
                     f'checkpoint (e.g. --doc-cap smaller than the run). '
                     f'Re-run with the original --doc-cap or delete the checkpoint.')
        V = [per_doc_layer[k] for k in keys]
        sl = _agg([v['s_l_median'] for v in V])
        sr = _agg([v['slope_ratio_median'] for v in V])
        qf = _agg([v['quad_frac_median'] for v in V])
        rfe = np.array([[v['resid_frac_by_eps_med'][e] for e in range(NEPS)]
                        for v in V])
        d3fix = {r: np.array([v['d3_by_r']['fixP'][str(r)][0] for v in V])
                 for r in R_SWEEP}
        d3raw = {r: np.array([v['d3_by_r']['raw'][str(r)][0] for v in V])
                 for r in R_SWEEP}
        d3gf = {r: np.array([v['d3_by_r']['gf'][str(r)][0] for v in V])
                for r in R_SWEEP}
        db = np.array([v['delta_b_rel'] for v in V
                       if v['delta_b_rel'] is not None])
        alphas = np.array([v['ols']['alpha'] for v in V
                           if v['ols'] and v['ols']['alpha'] is not None])
        def _boot(arr, stat='median'):
            idx = boot_rng.integers(0, len(arr), (BOOT_N, len(arr)))
            s = np.median(arr[idx], axis=1) if stat == 'median' \
                else arr[idx].mean(axis=1)
            return [float(np.percentile(s, 2.5)), float(np.percentile(s, 97.5))]
        sl_arr = np.array([v['s_l_median'] for v in V])
        d3b = {'fixP': {r: _boot(d3fix[r]) for r in R_SWEEP},
               'raw': {r: _boot(d3raw[r]) for r in R_SWEEP},
               'gf': {r: _boot(d3gf[r]) for r in R_SWEEP}}
        d3_by_r_out = {}
        for m in ('raw', 'gf', 'fixP'):
            arrm = d3fix if m == 'fixP' else (d3raw if m == 'raw' else d3gf)
            d3_by_r_out[m] = {}
            for r in R_SWEEP:
                arr = arrm[r]
                d3_by_r_out[m][str(r)] = {
                    'median': float(np.median(arr)),
                    'iqr': [float(np.percentile(arr, 25)),
                            float(np.percentile(arr, 75))],
                    'boot_ci': d3b[m][r]}
        layers_out[str(l)] = {
            's_l': {'median': sl[0], 'iqr': sl[1], 'boot_ci': _boot(sl_arr)},
            'a_vs_ESG_cos': _agg([v['a_vs_ESG_cos'] for v in V]),
            'slope_ratio': {'median': sr[0], 'iqr': sr[1]},
            'quad_frac': {'median': qf[0], 'iqr': qf[1]},
            'A1_median': _agg([v['A1_median'] for v in V]),
            'nb_median': _agg([v['nb_median'] for v in V]),
            'frac_c_sig': _agg([v['frac_c_sig'] for v in V]),
            'resid_frac_by_eps': {f'{int(round(EPS_REL[e] * 100))}pct':
                                  {'median': float(np.median(rfe[:, e])),
                                   'max_over_docs': float(np.max(rfe[:, e]))}
                                  for e in range(NEPS)},
            'd3_by_r': d3_by_r_out,
            'stratification': {
                'bQ1_fro': _agg([v['bQ1_fro'] for v in V]),
                'bQ4_fro': _agg([v['bQ4_fro'] for v in V]),
                'delta_b_rel': ({'median': float(np.median(db)),
                                 'iqr': [float(np.percentile(db, 25)),
                                         float(np.percentile(db, 75))],
                                 'boot_ci': _boot(db)} if len(db) else None),
                'ols': {'alpha': [float(np.median(alphas)),
                                  [float(np.percentile(alphas, 25)),
                                   float(np.percentile(alphas, 75))]]
                        if len(alphas) else None,
                        'pooled': _pooled_ols(cache, keys)},
                'n_docs_with_strata': int(len(db)),
            },
            'cross_checks': {k: _agg([v['cross_checks'][k] for v in V])
                             for k in ('rho_nb_dSVD_raw1', 'rho_nb_dSVD_fixP1',
                                       'rho_nb_dSVD_gf1', 'rho_nb_dp')},
            'sentinel_frac': _agg([v['sentinel_fraction'] for v in V]),
        }
        print(f'L{l}: s_l={sl[0]:.4f} slope={sr[0]:.4f} quad={qf[0]:.4f} '
              f'd3r1 raw/gf/fixP = '
              f'{np.median(d3raw[1]):.3f}/{np.median(d3gf[1]):.3f}/'
              f'{np.median(d3fix[1]):.3f} '
              f'd_b={np.median(db) if len(db) else None}', flush=True)
    report = {
        'meta': {
            'experiment': ('E-32r.3 (layer profile; supersedes E-32r.2 on '
                           'shared layers: normalized p0, L19 added, '
                           'T=128 all layers, GPU, float32 dz / float64 fit, '
                           'new quad_frac metric)'),
            'model': model_dir,
            'cache': cache_dir,
            'layers': list(a.layers),
            'n_docs': len(docs),
            'T': {str(l): T_FULL for l in a.layers},
            'eps_rel': [float(e) for e in EPS_REL],
            'eps_canon': EPS_CANON,
            'fit': 'dz = a*e + b*e^2 + c*e^3 (no const), eps absolute in h',
            's_l': 'eps_canon * ||off(b_t)||_F / ||off(ESG_raw_t)||_F (RAW norm)',
            'slope_ratio': ('E-32r.2 formula UNCHANGED: median of '
                            '|dz(e0)|/e0 / (|dz(e1)|/e1); 1.0 = linear regime'),
            'quad_frac': ('NEW (no E-32r.2 counterpart): (s1-s2)/|s2|, '
                          's1=||a1||, s2=||a1+2*(eps0*||h_t||)*a2||; '
                          '0 = quadratic term negligible at eps0'),
            'a_vs_ESG_cos': ('FIT-QUALITY sanity check (chain rule a1 = J e, '
                             'exact for any J): ~1 expected on every layer '
                             'with small resid_frac; NOT a depth metric'),
            'aggregation': 'E-34: per-doc median -> median/IQR over docs',
            'sentinel': 'P128_RAW.sum < 0.5, endpoints-only valid pairs; '
                        'sentinel_frac = share raw rows, nb_median over ~sent',
            'p0': 'canonical t0 position, NORMALIZED to sum 1',
            'd3': 'top-r right singular direction angle (cos-sim, exp33r)',
            'd3_fixP_note': ('diagnostics only on mid layers (TЗ §9: '
                             'J = M_K*P_u exact only at L23)'),
            'bootstrap': f'{BOOT_N} iterations over docs (seed=12345)',
            'device': DEVICE,
        },
        'layers': layers_out,
        'verdicts': {},
        'per_doc': {k: v for k, v in per_doc_layer.items()},
    }
    # --- verdicts ---
    sl_med = [layers_out[str(l)]['s_l']['median'] for l in a.layers]
    lin = {}
    for l in a.layers:
        vals = [layers_out[str(l)]['resid_frac_by_eps'][
            f'{int(round(EPS_REL[e] * 100))}pct']['median'] for e in range(NEPS)]
        lin[str(l)] = bool(np.median(vals).max() < 0.01)
    s_ok = all(v is not None and v < 0.3 for v in sl_med)
    report['verdicts'] = {
        's_l_lt_0.3_all_layers': bool(s_ok),
        's_l_medians': {str(l): sl_med[i] for i, l in enumerate(a.layers)},
        'resid_frac_lt_1pct_per_layer': lin,
        'linear_verdict': bool(s_ok and all(lin.values())),
        'depth_metrics': 's_l(l), d3_raw(1,l), d3_gf(1,l) — profile over '
                         'L6..L23; d3_fixP(1,l) diagnostics on mid layers',
        'note': ('A/B/C resolved via d3 sweep + stratification + cross-checks '
                 '(see layers.*)'),
    }
    os.makedirs(os.path.dirname(a.out) or '.', exist_ok=True)
    with open(a.out, 'w', encoding='utf-8') as f:
        json.dump(_clean(report), f, indent=1, ensure_ascii=False,
                  allow_nan=False)
    print(f'WROTE {a.out}  total {time.time() - t0:.0f}s', flush=True)


In [ ]:
# ---------- run exp32r3_layer_profile ----------
import gc
# Hard cleanup from previous kernel version (same session, different code)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
import inspect
sig = inspect.signature(main_32r3layerprofile)
kwargs = {}
params = list(sig.parameters)
if 'small_model' in params: kwargs['small_model'] = SMALL
if 'large_model' in params: kwargs['large_model'] = LARGE
if 'model' in params: kwargs['model'] = SMALL
if 'device_L' in params: kwargs['device_L'] = device_L
if 'device_S' in params: kwargs['device_S'] = device_S
if 'device' in params: kwargs['device'] = device_S
if 'results_dir' in params: kwargs['results_dir'] = RES
if 'targets_dir' in params: kwargs['targets_dir'] = os.path.join(RES, 'targets')
# Remove conflicting keys: if both model and small_model, prefer specific
if 'small_model' in kwargs and 'model' in kwargs: del kwargs['model']
# Memory diagnostic before experiment
if torch.cuda.is_available():
    print(f'cuda:0 alloc={torch.cuda.memory_allocated(0)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(0)/1024**3:.1f}GiB')
    print(f'cuda:1 alloc={torch.cuda.memory_allocated(1)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(1)/1024**3:.1f}GiB')
t0 = time.time()
try:
    main_32r3layerprofile(**kwargs)
    elapsed = time.time() - t0
    print(f"[OK] exp32r3_layer_profile {elapsed:.0f}s")
    # Persist models (in case experiment loaded from HF but didn't save)
    try:
        dst_s = os.path.join(WORKING, "qwen35-08b")
        dst_l = os.path.join(WORKING, "qwen35-4b")
        if os.path.isdir(dst_s):
            print(f"Small model persisted at {dst_s}")
        if os.path.isdir(dst_l):
            print(f"Large model persisted at {dst_l}")
    except Exception:
        pass
except Exception as e:
    print(f"[FAIL] exp32r3_layer_profile: {e}")
    traceback.print_exc()
print("\nResults:")
if os.path.isdir(RES):
    try:
        for fn in sorted(os.listdir(RES)):
            print(f"  {fn} {os.path.getsize(os.path.join(RES, fn)):,} bytes")
    except OSError:
        pass
